In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics 
from sklearn.metrics import classification_report, confusion_matrix
from tabulate import tabulate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.semi_supervised import SelfTrainingClassifier


In [4]:
# Load the datase
dataset = pd.read_csv("../data/pima.csv")

# Print the shape
shape = dataset.shape
print("Shape of the dataset:", shape)

# Display the first few rows
print(dataset.head())

Shape of the dataset: (768, 9)
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


In [5]:
#  Display the main statistical characteristics of each column
msc = dataset.describe()
print("Display the main statistical characteristics\n", msc)

# Identify columns where the minimum value takes an unreasonable value equal to zero
columns_zero = [col for col in dataset.columns if col != 'Outcome' and dataset[col].min() == 0]
print("Colums with unreasonable zero values:\n", columns_zero)

Display the main statistical characteristics
        Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  120.894531      69.105469      20.536458   79.799479   
std       3.369578   31.972618      19.355807      15.952218  115.244002   
min       0.000000    0.000000       0.000000       0.000000    0.000000   
25%       1.000000   99.000000      62.000000       0.000000    0.000000   
50%       3.000000  117.000000      72.000000      23.000000   30.500000   
75%       6.000000  140.250000      80.000000      32.000000  127.250000   
max      17.000000  199.000000     122.000000      99.000000  846.000000   

              BMI  DiabetesPedigreeFunction         Age     Outcome  
count  768.000000                768.000000  768.000000  768.000000  
mean    31.992578                  0.471876   33.240885    0.348958  
std      7.884160                  0.331329   11.760232    

In [6]:
# Replace zero values with the median of the corresponding column
for col in columns_zero:
    median_value = dataset[col].median()
    dataset[col] = dataset[col].replace(0, median_value)

# print updated dataset characteristics with Non zero values
msc_upd = dataset.describe()
print("Display the update statistical characteristics\n", msc_upd)

Display the update statistical characteristics
        Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      4.278646  121.656250      72.386719      27.334635   94.652344   
std       3.021516   30.438286      12.096642       9.229014  105.547598   
min       1.000000   44.000000      24.000000       7.000000   14.000000   
25%       2.000000   99.750000      64.000000      23.000000   30.500000   
50%       3.000000  117.000000      72.000000      23.000000   31.250000   
75%       6.000000  140.250000      80.000000      32.000000  127.250000   
max      17.000000  199.000000     122.000000      99.000000  846.000000   

              BMI  DiabetesPedigreeFunction         Age     Outcome  
count  768.000000                768.000000  768.000000  768.000000  
mean    32.450911                  0.471876   33.240885    0.348958  
std      6.875366                  0.331329   11.760232  

In [7]:


# Define features space (Fs) and target variable (t)
Fs = dataset.drop(columns=['Outcome'])
t= dataset['Outcome']

# Perform stratified split (700 for training, 68 for testing)
X_train, X_test, y_train, y_test = train_test_split(
    Fs, t, train_size=700, stratify=t, random_state=42
)

# check the datasets 
print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])

Training set size: 700
Test set size: 68


In [8]:
# Create Decision Tree classifer
dec_tree_2 = DecisionTreeClassifier(random_state=42)

# Train Decision Tree Classifer
dec_tree_2 = dec_tree_2.fit(X_train,y_train)

#Predict the response for test dataset
y_pred = dec_tree_2.predict(X_test)

# create the classification report and the confusion matrix
class_report = classification_report(y_test, y_pred, output_dict=True)
conf_matrix = confusion_matrix(y_test, y_pred)

# Extract the relevant metrics for class "1"
accuracy = class_report["accuracy"]
precision = class_report["1"]["precision"]
recall = class_report["1"]["recall"]
f1_score_1 = class_report["1"]["f1-score"]
    
# results
metrics_summary = [] #an empty df to store the data
metrics_summary.append([ accuracy, precision, recall, f1_score_1, conf_matrix])

# Print the results in table format
print(tabulate(metrics_summary, headers=[ "Accuracy", "Precision ", "Recall ", "F1-Score ", "Confusion Matrix"], tablefmt="fancy_grid"))

╒════════════╤══════════════╤═══════════╤═════════════╤════════════════════╕
│   Accuracy │   Precision  │   Recall  │   F1-Score  │ Confusion Matrix   │
╞════════════╪══════════════╪═══════════╪═════════════╪════════════════════╡
│   0.691176 │     0.565217 │  0.541667 │    0.553191 │ [[34 10]           │
│            │              │           │             │  [11 13]]          │
╘════════════╧══════════════╧═══════════╧═════════════╧════════════════════╛


The result is 69.1% accuracy, that is a acceptable number for a decision tree classifier

- A Random Forest classifier using the default values of its parameters.

In [9]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

# create the classification report and the confusion matrix
class_report = classification_report(y_test, y_pred_rf, output_dict=True)
conf_matrix = confusion_matrix(y_test, y_pred_rf)

# Extract the relevant metrics for class "1"
accuracy = class_report["accuracy"]
precision = class_report["1"]["precision"]
recall = class_report["1"]["recall"]
f1_score_1 = class_report["1"]["f1-score"]
    
# results
metrics_summary = [] #an empty df to store the data
metrics_summary.append([ accuracy, precision, recall, f1_score_1, conf_matrix])

# Print the results in table format
print(tabulate(metrics_summary, headers=[ "Accuracy", "Precision ", "Recall ", "F1-Score ", "Confusion Matrix"], tablefmt="fancy_grid"))

╒════════════╤══════════════╤═══════════╤═════════════╤════════════════════╕
│   Accuracy │   Precision  │   Recall  │   F1-Score  │ Confusion Matrix   │
╞════════════╪══════════════╪═══════════╪═════════════╪════════════════════╡
│   0.823529 │     0.730769 │  0.791667 │        0.76 │ [[37  7]           │
│            │              │           │             │  [ 5 19]]          │
╘════════════╧══════════════╧═══════════╧═════════════╧════════════════════╛


With the random forest classifier the accuracy score is 82.3% Accuracy is improved with this model in comparison with the last method the dicision tree classifier

- A Bagging classifier with an SVM classifier (linear kernel) and 10 estimators.

In [10]:
svm = BaggingClassifier(estimator=SVC(kernel="linear"), n_estimators=10, random_state=42)
svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)

# create the classification report and the confusion matrix
class_report = classification_report(y_test, y_pred_svm, output_dict=True)
conf_matrix = confusion_matrix(y_test, y_pred_svm)

# Extract the relevant metrics for class "1"
accuracy = class_report["accuracy"]
precision = class_report["1"]["precision"]
recall = class_report["1"]["recall"]
f1_score_1 = class_report["1"]["f1-score"]
    
# results
metrics_summary = [] #an empty df to store the data
metrics_summary.append([ accuracy, precision, recall, f1_score_1, conf_matrix])

# Print the results in table format
print(tabulate(metrics_summary, headers=[ "Accuracy", "Precision ", "Recall ", "F1-Score ", "Confusion Matrix"], tablefmt="fancy_grid"))

╒════════════╤══════════════╤═══════════╤═════════════╤════════════════════╕
│   Accuracy │   Precision  │   Recall  │   F1-Score  │ Confusion Matrix   │
╞════════════╪══════════════╪═══════════╪═════════════╪════════════════════╡
│   0.764706 │     0.666667 │  0.666667 │    0.666667 │ [[36  8]           │
│            │              │           │             │  [ 8 16]]          │
╘════════════╧══════════════╧═══════════╧═════════════╧════════════════════╛


- An AdaBoost classifier with a decision tree classifier, using 100 estimators and a learning rate of 0.25.

In [11]:
abc = AdaBoostClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, learning_rate=0.25, random_state=42)
abc.fit(X_train, y_train)

y_pred_abc = abc.predict(X_test)

# create the classification report and the confusion matrix
class_report = classification_report(y_test, y_pred_abc, output_dict=True)
conf_matrix = confusion_matrix(y_test, y_pred_abc)

# Extract the relevant metrics for class "1"
accuracy = class_report["accuracy"]
precision = class_report["1"]["precision"]
recall = class_report["1"]["recall"]
f1_score_1 = class_report["1"]["f1-score"]
    
# results
metrics_summary = [] #an empty df to store the data
metrics_summary.append([ accuracy, precision, recall, f1_score_1, conf_matrix])

# Print the results in table format
print(tabulate(metrics_summary, headers=[ "Accuracy", "Precision ", "Recall ", "F1-Score ", "Confusion Matrix"], tablefmt="fancy_grid"))

╒════════════╤══════════════╤═══════════╤═════════════╤════════════════════╕
│   Accuracy │   Precision  │   Recall  │   F1-Score  │ Confusion Matrix   │
╞════════════╪══════════════╪═══════════╪═════════════╪════════════════════╡
│   0.661765 │     0.521739 │       0.5 │    0.510638 │ [[33 11]           │
│            │              │           │             │  [12 12]]          │
╘════════════╧══════════════╧═══════════╧═════════════╧════════════════════╛


The random forest classifier is the most accurate model in comparison with the other three with Accuracy 82% and also has the highest precent of precision  with 73%. There is also a balance betwenn Precicion and Recall. The Bagging classifier with SVM it also perform well with 76% Accuracy but it is not outperforming Random Forests.  The Decision Tress are not performing so well like the first 2 model with accuracy of 69% and the lowest performance of previus 3, AdaBoost Classifier model has an accuracy 66%. The random forest look like the ideal choise for this dataset because is not only outperfoming the other model but handles the data better than the other, is not sensitive ti noise like Adaboost and is not overfitting the data like Dicission Trees.

In [12]:


# Randomly select 200 instances from the initial training set as labeled
np.random.seed(42)
labeled_indices = np.random.choice(X_train.index, size=200, replace=False)
unlabeled_indices = np.setdiff1d(X_train.index, labeled_indices)

# Create labeled and unlabeled datasets with loc funtion
X_labeled = X_train.loc[labeled_indices]
y_labeled = y_train.loc[labeled_indices]

X_unlabeled = X_train.loc[unlabeled_indices]
y_unlabeled = np.full_like(y_train.loc[unlabeled_indices], -1)

# Combine labeled and unlabeled data
X_combine = pd.concat([X_labeled, X_unlabeled])
y_combine = np.concatenate([y_labeled, y_unlabeled])

# Select the best model (Random Forest)
base_model = RandomForestClassifier(random_state=42)

# Define the Self-Training model with Random Forest as the base classifier
stm = SelfTrainingClassifier(base_model, criterion='threshold', threshold=0.99)

# Train the supervised model with the labeled data
supervised_model = RandomForestClassifier(random_state=42)
supervised_model.fit(X_labeled, y_labeled)

# Train the semi-supervised model with the combine data
stm.fit(X_combine, y_combine)

# Evaluate both models on the test set
y_pred_supervised = supervised_model.predict(X_test)
y_pred_combine = stm.predict(X_test)

# display classification reports
report_supervised = classification_report(y_test, y_pred_supervised, output_dict=True)
report_semi = classification_report(y_test, y_pred_combine, output_dict=True)


semi_supervised_results = {
    "Supervised Model (200 labeled)": report_supervised,
    "Semi-Supervised Model (Self-Training)": report_semi
}

semi_supervised_results


{'Supervised Model (200 labeled)': {'0': {'precision': 0.8536585365853658,
   'recall': 0.7954545454545454,
   'f1-score': 0.8235294117647058,
   'support': 44.0},
  '1': {'precision': 0.6666666666666666,
   'recall': 0.75,
   'f1-score': 0.7058823529411765,
   'support': 24.0},
  'accuracy': 0.7794117647058824,
  'macro avg': {'precision': 0.7601626016260162,
   'recall': 0.7727272727272727,
   'f1-score': 0.7647058823529411,
   'support': 68.0},
  'weighted avg': {'precision': 0.787661406025825,
   'recall': 0.7794117647058824,
   'f1-score': 0.7820069204152249,
   'support': 68.0}},
 'Semi-Supervised Model (Self-Training)': {'0': {'precision': 0.8095238095238095,
   'recall': 0.7727272727272727,
   'f1-score': 0.7906976744186046,
   'support': 44.0},
  '1': {'precision': 0.6153846153846154,
   'recall': 0.6666666666666666,
   'f1-score': 0.64,
   'support': 24.0},
  'accuracy': 0.7352941176470589,
  'macro avg': {'precision': 0.7124542124542125,
   'recall': 0.7196969696969697,
   '

From the result of the Supervised and the Semi-Supervised model we can see tha the supervised model is performing better with  77%  accuracy than the Semi-Superviced model with 73% accuracy. The Superviced moder is also performing better precision and recall score that the Semi-Superviced model. The supervised model was performing well with the 200 labeled data sample and the Semi-Supervised model didnt improve the performance with the unlabeled data as it was expection to do. Maybe the reson the the Semi-Supervised model was not increase the performace form the Superviced model was the very high threshold what we used. 